# 🛍️ Shopping Mall Customer Segmentation
## Member 1 — K-Means Clustering (Optimized)
**Algorithm:** K-Means Clustering  

### 📌 老师反馈优化点：
1. **Elbow Method 改进**：使用 `yellowbrick` 的 `KElbowVisualizer` 自动标注“拐点”，使图片更清晰、更专业。
2. **3D 可视化**：解决 2D 聚类重叠（Overlapping）的问题，通过 3D 视角更清晰地展示不同簇之间的界限。
3. **V-measure Score**：添加外部评估指标，衡量聚类结果与真实标签（如性别）的一致性。
4. **详细解释**：为每个部分提供业务意义说明。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, v_measure_score
from mpl_toolkits.mplot3d import Axes3D

# 尝试导入 yellowbrick，如果环境没有则使用传统方法
try:
    from yellowbrick.cluster import KElbowVisualizer
    HAS_YELLOWBRICK = True
except ImportError:
    HAS_YELLOWBRICK = False

import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported successfully!')

## 1. Load Pre-processed Data
**目的**：加载在 Step 0 中处理好的标准化数据。

In [ ]:
X_scaled = np.load('X_scaled.npy')
df_clean = pd.read_csv('data/data_preprocessed.csv')
df_original = pd.read_csv('data/Shopping Mall Customer Segmentation Data .csv')

print(f'✅ Loaded X_scaled shape: {X_scaled.shape}')

## 2. Optimal K — Elbow Method (Improved)
**意义**：Elbow Method（肘部法）通过观察“畸变程度”随 K 增加的下降速度来确定最佳聚类数。老师反馈之前的图不好看，我们现在使用更专业的可视化工具。

In [ ]:
model = KMeans(random_state=42)

if HAS_YELLOWBRICK:
    visualizer = KElbowVisualizer(model, k=(2,11))
    visualizer.fit(X_scaled)
    visualizer.show()
    best_k = visualizer.elbow_value_
else:
    # 传统方法作为备选
    inertias = []
    for k in range(2, 11):
        km = KMeans(n_clusters=k, random_state=42).fit(X_scaled)
        inertias.append(km.inertia_)
    
    plt.figure(figsize=(8, 5))
    plt.plot(range(2, 11), inertias, 'bo-')
    plt.xlabel('Number of Clusters (k)')
    plt.ylabel('Inertia')
    plt.title('Elbow Method for Optimal k')
    plt.show()
    best_k = 5 # 假设为 5

print(f"💡 意义：图中的 'Elbow'（肘部）位置即为最佳 K 值。在这个点之后，增加聚类数带来的收益（Inertia 下降）会显著变慢。")

## 3. Train K-Means Model
**目的**：使用确定的最佳 K 值进行聚类。

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42)
labels = kmeans.fit_predict(X_scaled)

df_clean['Cluster'] = labels
df_original['Cluster'] = labels
print(f"✅ K-Means trained with k={best_k}")

## 4. Evaluation: V-measure Score
**意义**：老师建议使用 V-measure。它衡量聚类结果与真实标签（如性别）的匹配程度。V-measure 结合了同质性（Homogeneity）和完整性（Completeness）。

In [ ]:
v_score = v_measure_score(df_original['Gender'], labels)
sil_score = silhouette_score(X_scaled, labels)

print(f"V-measure Score: {v_score:.4f}")
print(f"Silhouette Score: {sil_score:.4f}")

print("\n💡 意义：V-measure 越接近 1，说明聚类结果与性别的关联度越高。如果分数较低，说明性别并不是划分客户群体的主要依据，收入和消费行为更重要。")

## 5. 3D Cluster Visualization (解决 Overlapping 问题)
**意义**：老师反馈 2D 图中簇重叠严重。通过 3D 可视化（Age, Income, Spending Score），我们可以更清晰地看到不同群体在空间中的分离情况。

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(df_original['Age'], 
                     df_original['Annual Income'], 
                     df_original['Spending Score'], 
                     c=labels, cmap='viridis', s=20, alpha=0.6)

ax.set_xlabel('Age')
ax.set_ylabel('Annual Income')
ax.set_zlabel('Spending Score')
ax.set_title(f'3D K-Means Clustering (k={best_k})')

legend1 = ax.legend(*scatter.legend_elements(), title="Clusters")
ax.add_artist(legend1)

plt.show()

print("💡 意义：3D 图展示了客户在三个核心维度上的分布。通过旋转视角，可以清晰地看到原本在 2D 中重叠的点，在 3D 空间中其实是分属于不同高度或深度的簇。")